In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DataType

In [2]:
spark = SparkSession.builder \
    .appName("Optimizing Shuffle") \
    .master("local[*]") \
    .config("spark.cores.max",16) \
    .config("spark.executors.cores",4) \
    .config("spark.executors.memory", "512M") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 11:04:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Check spark default parallelism

spark.sparkContext.defaultParallelism

8

In [4]:
# Disable AQE amd Broadcast join
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enable", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [5]:
_schema = """
    first_name string,
    last_name string,
    job_title string,
    dob string,
    email string,
    phone string,
    salary float,
    department_id int
"""
emp = spark.read.format("csv").schema(_schema).option("header", True).load("/Users/AnhHuynh/Documents/PySpark/employee_records.txt")

emp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [11]:
emp_avg = emp.groupBy("department_id").agg(avg("salary").alias("avg_salary"))

In [12]:
# noop format reads the whole dataset without writing the data anywhere. This is to simulate Spark for benchmarking

emp_avg.write.format("noop").mode("overwrite").save()

In [22]:
spark.conf.set("spark.sql.shuffle.partitions",16)

In [23]:
emp.withColumn("partition_id", spark_partition_id()).where("partition_id = 0").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|partition_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|           0|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|           0|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|           0|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|           0|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffany

In [24]:
emp_part = spark.read.format("csv").schema(_schema).option("header", True).load("/Users/AnhHuynh/Documents/PySpark/employee_records.txt")

In [25]:
emp_avg = emp_part.groupBy("department_id").agg(avg("salary").alias("avg_salary"))

In [26]:
emp_avg.write.format("noop").mode("overwrite").save()